# Task 1: Data Exploration and Enrichment
**Objective:** Explore the starter dataset `ethiopia_fi_unified_data.csv`, inspect key indicators, validate the unified schema, and enrich the dataset with external observations, events, and impact links.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths Setup
DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "raw" / "ethiopia_fi_unified_data.csv"
REF_CODES_PATH = DATA_DIR / "raw" / "reference_codes.csv"
PROCESSED_DATA_PATH = DATA_DIR / "processed" / "ethiopia_fi_enriched.csv"

### 1. Load Starter Dataset & Reference Codes

In [3]:
df_unified = pd.read_csv(RAW_DATA_PATH)
df_ref = pd.read_csv(REF_CODES_PATH)

print(f"Unified Dataset Shape: {df_unified.shape}")
print(f"Reference Codes Shape: {df_ref.shape}")
df_unified.head()

Unified Dataset Shape: (43, 34)
Reference Codes Shape: (71, 4)


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


### 2. Explore Unified Schema & Breakdown

In [4]:
print("--- Record Type Counts ---")
print(df_unified['record_type'].value_counts())

print("\n--- Pillar Distribution ---")
print(df_unified['pillar'].value_counts(dropna=False))

print("\n--- Source Type Breakdown ---")
if 'source_type' in df_unified.columns:
    print(df_unified['source_type'].value_counts(dropna=False))

--- Record Type Counts ---
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

--- Pillar Distribution ---
pillar
ACCESS           16
USAGE            11
NaN              10
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64

--- Source Type Breakdown ---
source_type
operator      15
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64


### 3. Schema Rule Enforcement Check
- **Rule:** `event` records should NOT have pre-assigned pillars (to keep data unbiased).
- **Rule:** Relationships are specified via `impact_link` records using `parent_id`. 

In [5]:
invalid_events = df_unified[(df_unified['record_type'] == 'event') & (df_unified['pillar'].notna())]
print(f"Events with invalid pre-assigned pillars: {len(invalid_events)}")

# Clean if any exist
df_unified.loc[df_unified['record_type'] == 'event', 'pillar'] = None

Events with invalid pre-assigned pillars: 0


### 4. Enrich Dataset with External Observations, Events & Impact Links

In [6]:
new_records = [
    # --- NEW OBSERVATIONS ---
    {
        "record_id": "OBS_ENR_001",
        "record_type": "observation",
        "pillar": "Access",
        "indicator": "Mobile Money Account Ownership (% adults)",
        "indicator_code": "ACC_MM_ACCOUNT",
        "value_numeric": 19.4,
        "observation_date": "2025-01-01",
        "source_name": "World Bank Global Findex 2025 Update",
        "source_url": "https://microdata.worldbank.org/index.php/catalog/7901",
        "confidence": "High",
        "collected_by": "Team Lead",
        "collection_date": "2026-07-18",
        "notes": "Captures quadrupling of mobile money accounts from 4.7% (2021) to 19.4%."
    },
    {
        "record_id": "OBS_ENR_002",
        "record_type": "observation",
        "pillar": "Infrastructure",
        "indicator": "4G Population Coverage (%)",
        "indicator_code": "INF_4G_COVERAGE",
        "value_numeric": 70.8,
        "observation_date": "2025-07-01",
        "source_name": "Ethio Telecom FY24/25 Performance Report",
        "source_url": "https://www.ethiotelecom.et",
        "confidence": "High",
        "collected_by": "Data Specialist",
        "collection_date": "2026-07-18",
        "notes": "4G footprint expanded to 70.8%, critical digital payment enabler."
    },
    {
        "record_id": "OBS_ENR_003",
        "record_type": "observation",
        "pillar": "Usage",
        "indicator": "Telebirr Registered Users (Millions)",
        "indicator_code": "USG_TELEBIRR_USERS",
        "value_numeric": 54.8,
        "observation_date": "2025-06-01",
        "source_name": "Ethio Telecom FY24/25 Performance Summary",
        "source_url": "https://www.ethiotelecom.et",
        "confidence": "High",
        "collected_by": "Analyst",
        "collection_date": "2026-07-18",
        "notes": "Supply-side registered base to track against active demand-side Findex usage."
    },
    {
        "record_id": "OBS_ENR_004",
        "record_type": "observation",
        "pillar": "Infrastructure",
        "indicator": "Total Active Agent Outlets",
        "indicator_code": "INF_AGENTS_TOTAL",
        "value_numeric": 540000.0,
        "observation_date": "2025-06-01",
        "source_name": "Digital Finance Ethiopia Hub / NBE Reports",
        "source_url": "https://digitalfinance.shega.co",
        "confidence": "Medium",
        "collected_by": "Analyst",
        "collection_date": "2026-07-18",
        "notes": "Agent network coverage proxy for cash-in/cash-out liquidity access."
    },

    # --- NEW EVENTS ---
    {
        "record_id": "EVT_ENR_001",
        "record_type": "event",
        "pillar": None,  # Schema Rule: Events leave pillar empty
        "indicator": "EthSwitch Instant Payment System (EIPS) Go-Live",
        "indicator_code": "EVT_ETHSWITCH_IPS",
        "value_numeric": None,
        "observation_date": "2024-02-15",
        "source_name": "AfricaNenda SIIPS Report 2024",
        "source_url": "https://africanenda.org",
        "confidence": "High",
        "collected_by": "Policy Analyst",
        "collection_date": "2026-07-18",
        "notes": "Real-time national switch enabling instant cross-bank and PSP transfers."
    },
    {
        "record_id": "EVT_ENR_002",
        "record_type": "event",
        "pillar": None,  # Schema Rule: Events leave pillar empty
        "indicator": "National Bank Mandates Unified ETHQR Standard",
        "indicator_code": "EVT_ETHQR_MANDATE",
        "value_numeric": None,
        "observation_date": "2024-11-01",
        "source_name": "National Bank of Ethiopia Directives",
        "source_url": "https://nbe.gov.et",
        "confidence": "High",
        "collected_by": "Policy Analyst",
        "collection_date": "2026-07-18",
        "notes": "Standardized QR payment code mandatory for all financial institutions."
    },

    # --- NEW IMPACT LINKS ---
    {
        "record_id": "LNK_ENR_001",
        "record_type": "impact_link",
        "parent_id": "EVT_ENR_001",
        "pillar": "Access",
        "related_indicator": "ACC_MM_ACCOUNT",
        "impact_direction": "Positive",
        "impact_magnitude": 10.0,
        "lag_months": 12,
        "evidence_basis": "Empirical evidence from Kenya (M-Pesa) & Tanzania interbank interoperability deployment.",
        "source_name": "GSMA State of the Industry Report",
        "source_url": "https://gsma.com/sotir",
        "confidence": "Medium",
        "collected_by": "Modeling Lead",
        "collection_date": "2026-07-18",
        "notes": "Interoperability expands network utility, driving new wallet signups."
    },
    {
        "record_id": "LNK_ENR_002",
        "record_type": "impact_link",
        "parent_id": "EVT_ENR_002",
        "pillar": "Usage",
        "related_indicator": "USG_DIGITAL_PAYMENT",
        "impact_direction": "Positive",
        "impact_magnitude": 8.5,
        "lag_months": 6,
        "evidence_basis": "India UPI QR standard adoption and EthSwitch transactional volume acceleration.",
        "source_name": "EthSwitch S.C. Annual Report",
        "source_url": "https://ethswitch.com",
        "confidence": "High",
        "collected_by": "Modeling Lead",
        "collection_date": "2026-07-18",
        "notes": "Reduces merchant friction, boosting retail digital payment adoption."
    }
]

df_enriched = pd.concat([df_unified, pd.DataFrame(new_records)], ignore_index=True)
print(f"Total Enriched Dataset Count: {len(df_enriched)}")

Total Enriched Dataset Count: 51


### 5. Save Enriched Dataset to Processed Path

In [7]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_enriched.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Enriched dataset successfully saved to: {PROCESSED_DATA_PATH.resolve()}")

Enriched dataset successfully saved to: /home/mahlet/Documents/AI/ethiopia-fi-forecast/data/processed/ethiopia_fi_enriched.csv
